# Store Sales Forecasting - 03 PyTorch Neural Network

This notebook implements and trains a neural network in PyTorch for the supervised forecasting task defined in Part 02.

The model uses:

- categorical embeddings for variables such as `Store`, `StoreType`, and `Assortment`;
- standardized numerical features;
- a feed-forward neural network to predict `log1p(Sales)`;
- chronological validation using the prepared validation set;
- RMSPE and MAE evaluation on the original sales scale.

## 0. Environment Note

This notebook requires PyTorch. If the import cell below fails, install PyTorch in the same Python environment used by Jupyter.

A common CPU-only installation command is:

```bash
pip install torch
```

After installation, restart the notebook kernel and run the notebook again.

## 1. Imports and Settings

In [ ]:
from pathlib import Path
import json
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    import torch
    from torch import nn
    from torch.utils.data import Dataset, DataLoader
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        "PyTorch is not installed. Install it with `pip install torch`, restart the kernel, and rerun this notebook."
    ) from exc

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)

SEED = 42
BATCH_SIZE = 1024
EPOCHS = 20
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-5
DROPOUT = 0.15

# Set this to True for a very quick smoke test.
FAST_DEV_RUN = False

PREPARED_DIR = Path("outputs/prepared_notebook")
MODEL_DIR = Path("outputs/models")
PREDICTION_DIR = Path("outputs/predictions")

MODEL_DIR.mkdir(parents=True, exist_ok=True)
PREDICTION_DIR.mkdir(parents=True, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

## 2. Load Prepared Data

This notebook expects that Part 01 has already produced the prepared CSV files.

In [ ]:
train_path = PREPARED_DIR / "train_prepared.csv"
val_path = PREPARED_DIR / "validation_prepared.csv"
test_path = PREPARED_DIR / "test_prepared.csv"

missing_paths = [path for path in [train_path, val_path, test_path] if not path.exists()]
if missing_paths:
    raise FileNotFoundError(
        "Prepared data files are missing. Run 01_data_understanding_preparation.ipynb first.\n"
        + "\n".join(str(path) for path in missing_paths)
    )

train_df = pd.read_csv(train_path, parse_dates=["Date"], low_memory=False)
val_df = pd.read_csv(val_path, parse_dates=["Date"], low_memory=False)
test_df = pd.read_csv(test_path, parse_dates=["Date"], low_memory=False)

if FAST_DEV_RUN:
    train_df = train_df.sample(min(20000, len(train_df)), random_state=SEED).copy()
    val_df = val_df.sample(min(5000, len(val_df)), random_state=SEED).copy()

print("train:", train_df.shape)
print("validation:", val_df.shape)
print("test:", test_df.shape)
print("train period:", train_df["Date"].min().date(), "to", train_df["Date"].max().date())
print("validation period:", val_df["Date"].min().date(), "to", val_df["Date"].max().date())
print("test period:", test_df["Date"].min().date(), "to", test_df["Date"].max().date())

## 3. Define Target and Features

The model predicts `log1p(Sales)`. We exclude `Customers` because it is not available in the test set.

In [ ]:
target_column = "Sales"
training_target = "log_sales"

categorical_features = [
    "Store",
    "DayOfWeek",
    "StateHoliday",
    "StoreType",
    "Assortment",
]

numeric_features = [
    "Open",
    "Promo",
    "SchoolHoliday",
    "CompetitionDistance",
    "CompetitionOpenSinceMonth",
    "CompetitionOpenSinceYear",
    "Promo2",
    "Promo2SinceWeek",
    "Promo2SinceYear",
    "year",
    "month",
    "day",
    "weekofyear",
    "quarter",
    "dayofyear",
    "is_weekend",
    "is_month_start",
    "is_month_end",
    "days_since_start",
    "competition_open_months",
    "promo2_active",
    "dow_sin",
    "dow_cos",
    "month_sin",
    "month_cos",
]

model_features = categorical_features + numeric_features

for df_name, df, required_columns in [
    ("train_df", train_df, model_features + [target_column]),
    ("val_df", val_df, model_features + [target_column]),
    ("test_df", test_df, model_features),
]:
    missing = [column for column in required_columns if column not in df.columns]
    if missing:
        raise ValueError(f"Missing columns in {df_name}: {missing}")

train_df[training_target] = np.log1p(train_df[target_column].clip(lower=0))
val_df[training_target] = np.log1p(val_df[target_column].clip(lower=0))

print("Categorical features:", categorical_features)
print("Numeric features:", len(numeric_features))
print("Total input features:", len(model_features))

## 4. Encode Categorical Features

Neural networks cannot directly use text categories. We convert each category into an integer code. Code `0` is reserved for unknown categories.

In [ ]:
def as_category_text(series):
    return series.astype("string").fillna("__MISSING__").astype(str)


category_maps = {}
category_cardinalities = {}

for column in categorical_features:
    values = sorted(as_category_text(train_df[column]).unique())
    mapping = {"__UNKNOWN__": 0}
    mapping.update({value: index + 1 for index, value in enumerate(values)})
    category_maps[column] = mapping
    category_cardinalities[column] = len(mapping)


def encode_categorical_frame(df):
    encoded = pd.DataFrame(index=df.index)
    for column in categorical_features:
        mapping = category_maps[column]
        encoded[column] = as_category_text(df[column]).map(mapping).fillna(0).astype("int64")
    return encoded


train_cat = encode_categorical_frame(train_df)
val_cat = encode_categorical_frame(val_df)
test_cat = encode_categorical_frame(test_df)

category_summary = pd.DataFrame(
    {
        "feature": list(category_cardinalities.keys()),
        "n_categories_including_unknown": list(category_cardinalities.values()),
    }
)
display(category_summary)

## 5. Scale Numerical Features

The mean and standard deviation are computed only from the training set. This avoids using validation or test information during training.

In [ ]:
numeric_means = train_df[numeric_features].astype(float).mean()
numeric_stds = train_df[numeric_features].astype(float).std()
numeric_stds = numeric_stds.replace(0, 1).fillna(1)


def scale_numeric_frame(df):
    numeric = df[numeric_features].astype(float).copy()
    numeric = numeric.fillna(numeric_means)
    numeric = (numeric - numeric_means) / numeric_stds
    return numeric.astype("float32")


train_num = scale_numeric_frame(train_df)
val_num = scale_numeric_frame(val_df)
test_num = scale_numeric_frame(test_df)

display(train_num.head())

## 6. Create PyTorch Datasets and DataLoaders

In [ ]:
class StoreSalesDataset(Dataset):
    def __init__(self, categorical_data, numeric_data, target=None):
        self.categorical_data = torch.tensor(categorical_data.values, dtype=torch.long)
        self.numeric_data = torch.tensor(numeric_data.values, dtype=torch.float32)
        self.target = None if target is None else torch.tensor(target.values, dtype=torch.float32).view(-1, 1)

    def __len__(self):
        return len(self.numeric_data)

    def __getitem__(self, index):
        if self.target is None:
            return self.categorical_data[index], self.numeric_data[index]
        return self.categorical_data[index], self.numeric_data[index], self.target[index]


train_dataset = StoreSalesDataset(train_cat, train_num, train_df[training_target])
val_dataset = StoreSalesDataset(val_cat, val_num, val_df[training_target])
test_dataset = StoreSalesDataset(test_cat, test_num)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print("train batches:", len(train_loader))
print("validation batches:", len(val_loader))
print("test batches:", len(test_loader))

## 7. Define the Neural Network

Categorical features are passed through embedding layers. The embeddings are concatenated with the scaled numerical features and passed through a multilayer perceptron.

In [ ]:
def embedding_size(cardinality):
    return min(50, max(2, (cardinality + 1) // 2))


embedding_cardinalities = [category_cardinalities[column] for column in categorical_features]
embedding_dimensions = [embedding_size(cardinality) for cardinality in embedding_cardinalities]

print("Embedding cardinalities:", embedding_cardinalities)
print("Embedding dimensions:", embedding_dimensions)


class TabularSalesModel(nn.Module):
    def __init__(self, embedding_cardinalities, embedding_dimensions, n_numeric_features, dropout):
        super().__init__()
        self.embeddings = nn.ModuleList(
            [
                nn.Embedding(num_embeddings=cardinality, embedding_dim=dimension)
                for cardinality, dimension in zip(embedding_cardinalities, embedding_dimensions)
            ]
        )
        input_size = sum(embedding_dimensions) + n_numeric_features
        self.network = nn.Sequential(
            nn.Linear(input_size, 256),
            nn.ReLU(),
            nn.BatchNorm1d(256),
            nn.Dropout(dropout),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.BatchNorm1d(128),
            nn.Dropout(dropout),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 1),
        )

    def forward(self, categorical_inputs, numeric_inputs):
        embedded_features = [
            embedding_layer(categorical_inputs[:, index])
            for index, embedding_layer in enumerate(self.embeddings)
        ]
        x = torch.cat(embedded_features + [numeric_inputs], dim=1)
        return self.network(x)


model = TabularSalesModel(
    embedding_cardinalities=embedding_cardinalities,
    embedding_dimensions=embedding_dimensions,
    n_numeric_features=len(numeric_features),
    dropout=DROPOUT,
).to(device)

model

## 8. Metrics and Training Helpers

In [ ]:
def rmspe(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    mask = y_true != 0
    return np.sqrt(np.mean(((y_true[mask] - y_pred[mask]) / y_true[mask]) ** 2))


def mae(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return np.mean(np.abs(y_true - y_pred))


def predict_log_sales(model, loader):
    model.eval()
    predictions = []
    with torch.no_grad():
        for batch in loader:
            if len(batch) == 3:
                categorical_inputs, numeric_inputs, _ = batch
            else:
                categorical_inputs, numeric_inputs = batch
            categorical_inputs = categorical_inputs.to(device)
            numeric_inputs = numeric_inputs.to(device)
            outputs = model(categorical_inputs, numeric_inputs)
            predictions.append(outputs.cpu().numpy())
    return np.vstack(predictions).ravel()


def log_predictions_to_sales(log_predictions, open_values=None):
    sales = np.expm1(log_predictions)
    sales = np.clip(sales, 0, None)
    if open_values is not None:
        sales = np.where(np.asarray(open_values) == 0, 0, sales)
    return sales


criterion = nn.MSELoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

## 9. Train the Model

In [ ]:
history = []
best_val_rmspe = float("inf")
best_model_path = MODEL_DIR / "tabular_sales_nn.pt"

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_losses = []
    
    for categorical_inputs, numeric_inputs, targets in train_loader:
        categorical_inputs = categorical_inputs.to(device)
        numeric_inputs = numeric_inputs.to(device)
        targets = targets.to(device)
        
        optimizer.zero_grad()
        outputs = model(categorical_inputs, numeric_inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
        
        train_losses.append(loss.item())
    
    val_log_predictions = predict_log_sales(model, val_loader)
    val_sales_predictions = log_predictions_to_sales(val_log_predictions, val_df["Open"].values)
    val_rmspe = rmspe(val_df[target_column].values, val_sales_predictions)
    val_mae = mae(val_df[target_column].values, val_sales_predictions)
    train_loss = float(np.mean(train_losses))
    
    history.append(
        {
            "epoch": epoch,
            "train_mse_log_sales": train_loss,
            "val_rmspe": float(val_rmspe),
            "val_mae": float(val_mae),
        }
    )
    
    if val_rmspe < best_val_rmspe:
        best_val_rmspe = val_rmspe
        torch.save(model.state_dict(), best_model_path)
    
    print(
        f"Epoch {epoch:02d}/{EPOCHS} | "
        f"train MSE(log sales): {train_loss:.5f} | "
        f"val RMSPE: {val_rmspe:.5f} | "
        f"val MAE: {val_mae:.2f}"
    )

history_df = pd.DataFrame(history)
print("Best validation RMSPE:", best_val_rmspe)

## 10. Plot Training History

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history_df["epoch"], history_df["train_mse_log_sales"], marker="o")
axes[0].set_title("Training loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("MSE on log1p(Sales)")
axes[0].grid(True, alpha=0.3)

axes[1].plot(history_df["epoch"], history_df["val_rmspe"], marker="o", color="tab:orange")
axes[1].set_title("Validation RMSPE")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("RMSPE")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 11. Evaluate the Best Model on Validation Data

In [ ]:
model.load_state_dict(torch.load(best_model_path, map_location=device))

val_log_predictions = predict_log_sales(model, val_loader)
val_sales_predictions = log_predictions_to_sales(val_log_predictions, val_df["Open"].values)

validation_results = {
    "rmspe": float(rmspe(val_df[target_column].values, val_sales_predictions)),
    "mae": float(mae(val_df[target_column].values, val_sales_predictions)),
}

validation_results

In [ ]:
validation_plot_df = val_df[["Date", "Store", "Sales", "Open"]].copy()
validation_plot_df["prediction"] = val_sales_predictions

daily_validation = (
    validation_plot_df.groupby("Date", as_index=False)[["Sales", "prediction"]]
    .mean()
    .sort_values("Date")
)

plt.figure(figsize=(12, 4))
plt.plot(daily_validation["Date"], daily_validation["Sales"], label="Actual average Sales", linewidth=2)
plt.plot(daily_validation["Date"], daily_validation["prediction"], label="Neural network prediction", linewidth=2)
plt.title("Validation Period: Actual Sales vs Neural Network Prediction")
plt.xlabel("Date")
plt.ylabel("Average Sales")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 12. Compare With the Part 02 Baseline

If the Part 02 summary file exists, we can compare the neural network with the simple store/day-of-week average baseline.

In [ ]:
baseline_summary_path = PREPARED_DIR / "part_02_supervised_task_summary.json"

comparison_rows = [
    {
        "model": "PyTorch neural network",
        "validation_rmspe": validation_results["rmspe"],
        "validation_mae": validation_results["mae"],
    }
]

if baseline_summary_path.exists():
    with open(baseline_summary_path, "r", encoding="utf-8") as f:
        part_02_summary = json.load(f)
    baseline_results = part_02_summary.get("baseline_results", {})
    comparison_rows.insert(
        0,
        {
            "model": "Store + day-of-week baseline",
            "validation_rmspe": baseline_results.get("validation_rmspe"),
            "validation_mae": baseline_results.get("validation_mae"),
        },
    )
else:
    print("Part 02 baseline summary not found. Run 02_supervised_learning_task.ipynb to include the baseline comparison.")

comparison_df = pd.DataFrame(comparison_rows)
display(comparison_df)

## 13. Predict the Test Set and Save a Submission File

In [ ]:
test_log_predictions = predict_log_sales(model, test_loader)
test_sales_predictions = log_predictions_to_sales(test_log_predictions, test_df["Open"].values)

if "Id" in test_df.columns:
    submission_df = pd.DataFrame({"Id": test_df["Id"].values, "Sales": test_sales_predictions})
else:
    submission_df = test_df[["Store", "Date"]].copy()
    submission_df["Sales"] = test_sales_predictions

submission_path = PREDICTION_DIR / "submission_pytorch_nn.csv"
submission_df.to_csv(submission_path, index=False)

print("Saved submission to:", submission_path.resolve())
display(submission_df.head())

## 14. Save Modeling Artifacts

The model weights are saved separately from the preprocessing metadata. The metadata records feature lists, category mappings, scaling values, hyperparameters, and validation results.

In [ ]:
artifact_summary = {
    "model_path": str(best_model_path),
    "submission_path": str(submission_path),
    "target_column": target_column,
    "training_target": training_target,
    "categorical_features": categorical_features,
    "numeric_features": numeric_features,
    "category_maps": category_maps,
    "numeric_means": {column: float(value) for column, value in numeric_means.items()},
    "numeric_stds": {column: float(value) for column, value in numeric_stds.items()},
    "embedding_cardinalities": embedding_cardinalities,
    "embedding_dimensions": embedding_dimensions,
    "hyperparameters": {
        "batch_size": BATCH_SIZE,
        "epochs": EPOCHS,
        "learning_rate": LEARNING_RATE,
        "weight_decay": WEIGHT_DECAY,
        "dropout": DROPOUT,
    },
    "validation_results": validation_results,
}

artifact_path = MODEL_DIR / "tabular_sales_nn_metadata.json"
with open(artifact_path, "w", encoding="utf-8") as f:
    json.dump(artifact_summary, f, indent=2)

history_path = MODEL_DIR / "tabular_sales_nn_history.csv"
history_df.to_csv(history_path, index=False)

validation_predictions_path = PREDICTION_DIR / "validation_predictions_pytorch_nn.csv"
validation_plot_df.to_csv(validation_predictions_path, index=False)

print("Saved model weights:", best_model_path.resolve())
print("Saved metadata:", artifact_path.resolve())
print("Saved training history:", history_path.resolve())
print("Saved validation predictions:", validation_predictions_path.resolve())

## 15. What This Notebook Achieves

This notebook completes the first neural-network model for the forecasting task:

- categorical variables are encoded with embeddings;
- numerical variables are standardized using training-set statistics;
- the model is trained to predict `log1p(Sales)`;
- predictions are converted back to the sales scale;
- validation is performed on a future time window;
- test predictions are saved in a submission-style CSV.

In Part 04, we can improve this model with more advanced experiments such as lag features, rolling averages, ablation studies, or different neural-network architectures.